# NLP sur texte des effets — Synergies textuelles

Objectif : exploiter le champ `desc` (texte d'effet) des 13 797 cartes pour détecter des synergies **sans avoir besoin de decklists**.

Trois approches combinées :
1. **Graphe de références explicites** — si le texte de A cite B entre guillemets → lien direct
2. **Tags mécaniques** — mots-clés d'effet YGO (Banish, Negate, Tuner…) → cartes partageant les mêmes mécaniques
3. **TF-IDF cosine similarity** — similarité textuelle entre effets → paires textuellement proches

Résultat : table `text_synergies` + visualisations.

In [ ]:
import sqlite3
import re
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from pyvis.network import Network
import warnings
warnings.filterwarnings('ignore')

con = sqlite3.connect('../data/yugioh.db')

cards = pd.read_sql("""
    SELECT id, name, archetype, type, race, attribute, ban_tcg, views_week, desc
    FROM cards
    WHERE desc IS NOT NULL AND desc != ''
""", con)

print(f'Cartes chargées : {len(cards):,}')
print(f'Longueur desc   : {cards.desc.str.len().mean():.0f} chars (moy)')
print(f'Avec archetype  : {cards.archetype.notna().sum():,}')

## Phase 1 — Graphe de références explicites

Dans YGO, les noms de cartes et d'archetypes apparaissent **entre guillemets doubles** dans les textes d'effets.  
Ex : `"Branded"`, `"Ash Blossom & Joyous Spring"`, `"Fallen of Albaz"`

On construit un graphe dirigé : A → B si le texte de A cite B.

In [ ]:
# Index de recherche : nom exact + archetype
card_names = set(cards['name'].str.strip())
archetypes  = set(cards['archetype'].dropna().str.strip())

def extract_refs(text, card_names, archetypes):
    """Retourne (card_refs, archetype_refs) depuis un texte d'effet."""
    quoted = re.findall(r'"([^"]+)"', str(text))
    c_refs = [q for q in quoted if q in card_names]
    a_refs = [q for q in quoted if q in archetypes and q not in card_names]
    return c_refs, a_refs

cards[['card_refs', 'arch_refs']] = cards['desc'].apply(
    lambda t: pd.Series(extract_refs(t, card_names, archetypes))
)
cards['n_refs'] = cards['card_refs'].apply(len) + cards['arch_refs'].apply(len)

print(f'Cartes avec au moins 1 référence : {(cards.n_refs > 0).sum():,}')
print(f'Cartes avec 3+ références        : {(cards.n_refs >= 3).sum():,}')
print()
print('Top 10 cartes les plus référencées (citées dans d\'autres effets) :')
all_cited = [r for refs in cards['card_refs'] for r in refs]
cited_counts = pd.Series(all_cited).value_counts().head(10)
print(cited_counts.to_string())

In [ ]:
# Construire les arcs du graphe de références
# Arc (A → B) : texte de A cite le nom exact de B
# Poids : normalisé par le nombre de références de A (évite de surpondérer les longues cartes)
ref_edges = []

for _, row in cards.iterrows():
    if not row['card_refs']:
        continue
    unique_refs = list(set(row['card_refs']) - {row['name']})  # exclure auto-références
    if not unique_refs:
        continue
    w = round(1.0 / len(unique_refs), 4)  # poids uniforme partagé entre toutes les références
    for target in unique_refs:
        ref_edges.append({
            'card_a': row['name'],
            'card_b': target,
            'ref_weight': w,
        })

ref_df = pd.DataFrame(ref_edges)

# Agréger : si A cite B plusieurs fois (dans différentes cartes du même archetype), sommer
ref_df = ref_df.groupby(['card_a', 'card_b'])['ref_weight'].sum().reset_index()
ref_df['ref_weight'] = ref_df['ref_weight'].clip(upper=1.0).round(4)

print(f'Arcs de références : {len(ref_df):,}')
print()
print('Top 15 paires par poids de référence :')
print(ref_df.sort_values('ref_weight', ascending=False).head(15).to_string(index=False))

## Phase 2 — Tags mécaniques

Chaque carte est taguée avec les mécaniques YGO présentes dans son effet.  
Deux cartes partageant des mécaniques = synergie potentielle de gameplay.

On calcule ensuite le **Jaccard des tags** entre toutes les paires de cartes.

In [ ]:
MECHANIC_PATTERNS = {
    'special_summon':   r'special summon',
    'add_to_hand':      r'add .{1,40} to your hand',
    'banish':           r'\bbanish\b',
    'send_to_gy':       r'send .{1,40} to (the |your )?g\.?y|graveyard',
    'negate':           r'\bnegate\b',
    'draw':             r'\bdraw\b .{0,15}card',
    'destroy':          r'\bdestroy\b',
    'tuner':            r'\btuner\b',
    'quick_effect':     r'quick effect',
    'once_per_turn':    r'once per turn',
    'fusion':           r'\bfusion (summon|monster|material)\b',
    'synchro':          r'\bsynchro (summon|monster|material)\b',
    'xyz':              r'\bxyz (summon|monster|material)\b',
    'link':             r'\blink (summon|monster|material)\b',
    'ritual':           r'\britual (summon|monster|spell)\b',
    'pendulum':         r'pendulum (effect|scale|summon)',
    'bounce':           r'return .{1,40} (to|into) (the |your )?hand',
    'search_deck':      r'(from|in) (your |the )?deck',
    'counter':          r'counter trap|place a counter|counter on',
    'token':            r'\btoken(s)?\b',
    'tribute':          r'\btribute\b',
    'flip':             r'\bflip (effect|summon)\b',
    'atk_modifier':     r'gains? \d+ atk|loses? \d+ atk',
    'opponent_hand':    r"opponent'?s? hand",
    'gy_recursion':     r'from (the |your )?g\.?y|graveyard',
}

def tag_card(text):
    t = str(text).lower()
    return frozenset(k for k, p in MECHANIC_PATTERNS.items() if re.search(p, t))

cards['tags'] = cards['desc'].apply(tag_card)
cards['n_tags'] = cards['tags'].apply(len)

# Distribution des tags
tag_counts = pd.Series([t for tags in cards['tags'] for t in tags]).value_counts()
print('Distribution des tags mécaniques :')
print(tag_counts.to_string())

In [ ]:
# Jaccard des tags entre cartes du même archetype (beaucoup plus pertinent que cross-archetype)
# Pour les synergies cross-archetype, on utilisera le TF-IDF (Phase 3)

def keyword_jaccard(tags_a, tags_b):
    if not tags_a or not tags_b:
        return 0.0
    inter = len(tags_a & tags_b)
    union = len(tags_a | tags_b)
    return round(inter / union, 4) if union > 0 else 0.0

# Calculer par archetype pour rester à taille gérable
kw_rows = []
archs_with_cards = cards[cards.archetype.notna()].groupby('archetype')

for arch, grp in archs_with_cards:
    if len(grp) < 2:
        continue
    rows = grp.reset_index(drop=True)
    for i in range(len(rows)):
        for j in range(i+1, len(rows)):
            jac = keyword_jaccard(rows.loc[i, 'tags'], rows.loc[j, 'tags'])
            if jac >= 0.4:  # seuil : 40% de méchaniques en commun
                kw_rows.append({
                    'card_a': rows.loc[i, 'name'],
                    'card_b': rows.loc[j, 'name'],
                    'archetype': arch,
                    'kw_jaccard': jac,
                    'shared_keywords': sorted(rows.loc[i, 'tags'] & rows.loc[j, 'tags']),
                })

kw_df = pd.DataFrame(kw_rows).sort_values('kw_jaccard', ascending=False)
print(f'Paires intra-archetype Jaccard ≥ 0.4 : {len(kw_df):,}')
print()
print('Top 15 paires par Jaccard mécaniques :')
print(kw_df[['card_a','card_b','archetype','kw_jaccard','shared_keywords']].head(15).to_string(index=False))

## Phase 3 — TF-IDF Cosine Similarity

On vectorise les textes d'effets et on calcule la similarité cosinus entre toutes les cartes.  
Particulièrement utile pour détecter des synergies **cross-archetype** (cartes de styles similaires).

In [ ]:
# Preprocessing : lowercase, supprimer les noms de cartes propres (trop spécifiques)
# pour que le TF-IDF capture les mécaniques, pas les noms propres
def preprocess(text):
    t = str(text).lower()
    t = re.sub(r'"[^"]+"', 'CARDREF', t)  # remplacer les noms entre guillemets par un token neutre
    t = re.sub(r'\d+', 'NUM', t)           # normaliser les nombres
    t = re.sub(r'[^a-z\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

cards['desc_clean'] = cards['desc'].apply(preprocess)

# TF-IDF avec n-grams (1,2) pour capturer "special summon", "quick effect", etc.
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,           # ignorer les termes trop rares
    max_df=0.85,        # ignorer les termes trop communs ("once per turn" est partout)
    sublinear_tf=True,  # log(tf) pour réduire l'impact des répétitions
    max_features=3000,
)

tfidf_matrix = vectorizer.fit_transform(cards['desc_clean'])
print(f'Matrice TF-IDF : {tfidf_matrix.shape[0]} cartes × {tfidf_matrix.shape[1]} features')
print()
print('Top features TF-IDF :')
feat_names = vectorizer.get_feature_names_out()
mean_idf = np.array(tfidf_matrix.mean(axis=0)).flatten()
top_feat = pd.Series(mean_idf, index=feat_names).sort_values(ascending=False).head(30)
print(top_feat.to_string())

In [ ]:
# Similarité cosinus — calculée par batch pour éviter OOM (13k × 13k = 169M paires)
# On garde uniquement les paires avec similarité >= 0.5

TFIDF_THRESHOLD = 0.5
BATCH_SIZE = 500

n = tfidf_matrix.shape[0]
tfidf_rows = []

for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch = tfidf_matrix[start:end]
    sims = cosine_similarity(batch, tfidf_matrix)  # (batch_size × n)
    
    for i_local, i_global in enumerate(range(start, end)):
        row = sims[i_local]
        # Garder seulement les j > i_global pour éviter les doublons
        for j in np.where((row >= TFIDF_THRESHOLD) & (np.arange(n) > i_global))[0]:
            tfidf_rows.append({
                'card_a': cards.iloc[i_global]['name'],
                'card_b': cards.iloc[j]['name'],
                'tfidf_sim': round(float(row[j]), 4),
            })
    
    if start % 2000 == 0:
        print(f'  {start}/{n} lignes traitées… ({len(tfidf_rows):,} paires trouvées)')

tfidf_df = pd.DataFrame(tfidf_rows).sort_values('tfidf_sim', ascending=False)
print(f'\nPaires TF-IDF ≥ {TFIDF_THRESHOLD} : {len(tfidf_df):,}')
print()
print('Top 20 paires par similarité TF-IDF :')
print(tfidf_df.head(20).to_string(index=False))

In [ ]:
# Top paires cross-archetype (les plus intéressantes)
arch_map = cards.set_index('name')['archetype'].to_dict()

tfidf_df['arch_a'] = tfidf_df['card_a'].map(arch_map)
tfidf_df['arch_b'] = tfidf_df['card_b'].map(arch_map)
tfidf_df['cross'] = tfidf_df['arch_a'] != tfidf_df['arch_b']

print('Top 20 paires TF-IDF cross-archetype (synergies potentielles entre archetypes) :')
cross = tfidf_df[tfidf_df['cross'] & tfidf_df['arch_a'].notna() & tfidf_df['arch_b'].notna()]
print(cross[['card_a','arch_a','card_b','arch_b','tfidf_sim']].head(20).to_string(index=False))

## Phase 4 — Table text_synergies consolidée

On fusionne les 3 signaux en une table unique avec un **score composite** :
- `ref_score` : poids de référence explicite (Phase 1)
- `kw_jaccard` : Jaccard mécaniques (Phase 2)
- `tfidf_sim` : cosine TF-IDF (Phase 3)
- `text_synergy_score` = `0.5 × ref_score + 0.25 × kw_jaccard + 0.25 × tfidf_sim`

In [ ]:
# Normaliser les paires (card_a < card_b alphabétiquement pour éviter les doublons)
def normalize_pair(a, b):
    return (a, b) if a <= b else (b, a)

# Ref edges
ref_norm = ref_df.copy()
ref_norm[['card_a','card_b']] = ref_norm.apply(
    lambda r: pd.Series(normalize_pair(r['card_a'], r['card_b'])), axis=1
)
ref_norm = ref_norm.groupby(['card_a','card_b'])['ref_weight'].max().reset_index()
ref_norm = ref_norm.rename(columns={'ref_weight': 'ref_score'})

# KW edges
kw_norm = kw_df[['card_a','card_b','kw_jaccard','shared_keywords']].copy()
kw_norm[['card_a','card_b']] = kw_norm.apply(
    lambda r: pd.Series(normalize_pair(r['card_a'], r['card_b'])), axis=1
)
kw_norm = kw_norm.groupby(['card_a','card_b']).agg(
    kw_jaccard=('kw_jaccard','max'),
    shared_keywords=('shared_keywords','first')
).reset_index()
kw_norm['shared_keywords'] = kw_norm['shared_keywords'].apply(
    lambda x: ','.join(x) if isinstance(x, list) else ''
)

# TF-IDF edges
tfidf_norm = tfidf_df[['card_a','card_b','tfidf_sim']].copy()
tfidf_norm[['card_a','card_b']] = tfidf_norm.apply(
    lambda r: pd.Series(normalize_pair(r['card_a'], r['card_b'])), axis=1
)
tfidf_norm = tfidf_norm.groupby(['card_a','card_b'])['tfidf_sim'].max().reset_index()

# Merge all
merged = (
    ref_norm
    .merge(kw_norm, on=['card_a','card_b'], how='outer')
    .merge(tfidf_norm, on=['card_a','card_b'], how='outer')
)
merged = merged.fillna({'ref_score': 0.0, 'kw_jaccard': 0.0, 'tfidf_sim': 0.0, 'shared_keywords': ''})

# Score composite
merged['text_synergy_score'] = (
    0.5 * merged['ref_score'] +
    0.25 * merged['kw_jaccard'] +
    0.25 * merged['tfidf_sim']
).round(4)

# Ajouter archetype
merged['arch_a'] = merged['card_a'].map(arch_map)
merged['arch_b'] = merged['card_b'].map(arch_map)

# Filtrer les paires trop faibles
merged = merged[merged['text_synergy_score'] >= 0.1].copy()
merged = merged.sort_values('text_synergy_score', ascending=False).reset_index(drop=True)

print(f'Paires text_synergies (score ≥ 0.1) : {len(merged):,}')
print()
print('Top 20 par score composite :')
print(merged[['card_a','card_b','ref_score','kw_jaccard','tfidf_sim','text_synergy_score']].head(20).to_string(index=False))

In [ ]:
# Synergies intra-archetype
print('=== Top synergies intra-archetype ===')
intra = merged[merged['arch_a'] == merged['arch_b']].copy()
print(f'Paires intra-archetype : {len(intra):,}')
print()
for arch in ['Kewl Tune', 'Branded', 'Tenpai Dragon', 'Ryzeal']:
    sub = intra[intra['arch_a'] == arch].head(5)
    if not sub.empty:
        print(f'\n--- {arch} ---')
        print(sub[['card_a','card_b','text_synergy_score','shared_keywords']].to_string(index=False))

In [ ]:
# Synergies cross-archetype (les plus intéressantes pour le signal boutiques)
print('=== Top synergies cross-archetype ===')
cross_arch = merged[
    (merged['arch_a'] != merged['arch_b']) &
    merged['arch_a'].notna() & merged['arch_b'].notna()
].copy()
print(f'Paires cross-archetype : {len(cross_arch):,}')
print()
print(cross_arch[['card_a','arch_a','card_b','arch_b','text_synergy_score']].head(20).to_string(index=False))

## Phase 5 — Sauvegarde en base

In [ ]:
con2 = sqlite3.connect('../data/yugioh.db')

con2.execute("DROP TABLE IF EXISTS text_synergies")
con2.execute("""
    CREATE TABLE text_synergies (
        card_a              TEXT,
        card_b              TEXT,
        ref_score           REAL,
        kw_jaccard          REAL,
        tfidf_sim           REAL,
        text_synergy_score  REAL,
        shared_keywords     TEXT,
        arch_a              TEXT,
        arch_b              TEXT,
        PRIMARY KEY (card_a, card_b)
    )
""")

cols = ['card_a','card_b','ref_score','kw_jaccard','tfidf_sim','text_synergy_score','shared_keywords','arch_a','arch_b']
merged[cols].to_sql('text_synergies', con2, if_exists='append', index=False)

# Tags mécaniques par carte
con2.execute("DROP TABLE IF EXISTS card_mechanic_tags")
con2.execute("""
    CREATE TABLE card_mechanic_tags (
        card_name   TEXT PRIMARY KEY,
        archetype   TEXT,
        tags        TEXT,
        n_tags      INTEGER
    )
""")
tags_df = cards[['name','archetype','tags','n_tags']].copy()
tags_df['tags'] = tags_df['tags'].apply(lambda t: ','.join(sorted(t)))
tags_df = tags_df.rename(columns={'name': 'card_name'})
tags_df.to_sql('card_mechanic_tags', con2, if_exists='append', index=False)

con2.commit()
con2.close()

print(f'✓ {len(merged):,} paires sauvegardées dans text_synergies')
print(f'✓ {len(tags_df):,} cartes taguées dans card_mechanic_tags')

## Phase 6 — Visualisation : graphe de références d'un archetype

In [ ]:
def plot_text_synergy_graph(archetype, min_score=0.15, output_path=None):
    """Graphe des synergies textuelles pour un archetype donné."""
    con3 = sqlite3.connect('../data/yugioh.db')
    
    # Cartes de l'archetype
    arch_cards = pd.read_sql(
        "SELECT name FROM cards WHERE archetype = ?", con3, params=[archetype]
    )['name'].tolist()
    
    if not arch_cards:
        print(f'Archetype "{archetype}" non trouvé.')
        return
    
    # Paires impliquant au moins une carte de l'archetype
    pairs = pd.read_sql("""
        SELECT card_a, card_b, text_synergy_score, ref_score, shared_keywords, arch_a, arch_b
        FROM text_synergies
        WHERE text_synergy_score >= ?
    """, con3, params=[min_score])
    con3.close()

    arch_set = set(arch_cards)
    pairs = pairs[(pairs['card_a'].isin(arch_set)) | (pairs['card_b'].isin(arch_set))]
    
    if pairs.empty:
        print(f'Aucune paire pour {archetype} avec score ≥ {min_score}')
        return

    G = nx.Graph()
    for _, r in pairs.iterrows():
        G.add_edge(r['card_a'], r['card_b'], weight=r['text_synergy_score'])

    net = Network(height='700px', width='100%', bgcolor='#1a1a2e', font_color='white')
    net.from_nx(G)

    for node in net.nodes:
        name = node['id']
        if name in arch_set:
            node['color'] = '#e94560'
            node['size'] = 20
        else:
            node['color'] = '#0f3460'
            node['size'] = 12
        node['title'] = name
        node['label'] = name[:25] + '…' if len(name) > 25 else name

    net.set_options("""
    var options = {
      "physics": {"forceAtlas2Based": {"springLength": 120}, "solver": "forceAtlas2Based"},
      "edges": {"smooth": {"type": "continuous"}}
    }
    """)

    path = output_path or f'../data/graph_text_{archetype.lower().replace(" ","_")}.html'
    net.save_graph(path)
    print(f'Graphe sauvegardé : {path}')
    print(f'Noeuds : {G.number_of_nodes()} | Arcs : {G.number_of_edges()}')
    return net


plot_text_synergy_graph('Kewl Tune', min_score=0.15)
plot_text_synergy_graph('Branded', min_score=0.15)
plot_text_synergy_graph('Tenpai Dragon', min_score=0.15)

In [ ]:
# Bonus : cartes les plus "centrales" dans le graphe de synergies textuelles
# = cartes qui connectent le plus d'autres cartes par leurs effets

con4 = sqlite3.connect('../data/yugioh.db')
all_pairs = pd.read_sql(
    "SELECT card_a, card_b, text_synergy_score FROM text_synergies WHERE text_synergy_score >= 0.2",
    con4
)
con4.close()

G_full = nx.Graph()
for _, r in all_pairs.iterrows():
    G_full.add_edge(r['card_a'], r['card_b'], weight=r['text_synergy_score'])

degree = pd.Series(dict(G_full.degree(weight='weight')))
top_central = degree.sort_values(ascending=False).head(25)

print('Top 25 cartes les plus connectées textuellement (degree pondéré) :')
for card, deg in top_central.items():
    arch = arch_map.get(card, '—')
    print(f'  {deg:.2f}  {card}  [{arch}]')